# Revenue at Risk: a win-back diagnostic for a UK online retailer
**A mock Adobe Solution Consulting engagement.** I take two years of real transactions from a UK online gift retailer, find the high-value customers who are going quiet, put a £ figure on what they are worth, and show how Adobe Experience Cloud and its AI agents would win them back.

*Data: UCI "Online Retail II", about 1M real transactions, Dec 2009 to Dec 2011.*

**How to run:** Runtime → Run all. Loading takes 1–2 minutes.

In [ ]:
# 1. LOAD THE DATA
import os, glob, pandas as pd, numpy as np, matplotlib.pyplot as plt

URL = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"

def find_file():
    files = glob.glob("**/*online_retail*.xlsx", recursive=True) + glob.glob("**/*online_retail*.csv", recursive=True)
    return files[0] if files else None

if find_file() is None:
    os.system(f"wget -q -O retail.zip '{URL}' && unzip -o -q retail.zip")
if find_file() is None:
    # Fallback: download the dataset yourself (UCI or Kaggle), then upload the .xlsx or .csv here
    from google.colab import files
    files.upload()

path = find_file()
if path.endswith(".xlsx"):
    raw = pd.concat(pd.read_excel(path, sheet_name=None).values(), ignore_index=True)
else:
    raw = pd.read_csv(path, encoding="ISO-8859-1")
print(f"Loaded {len(raw):,} rows from {path}")

In [ ]:
# 2. CLEAN
df = raw.rename(columns={"Customer ID": "CustomerID", "Invoice": "Invoice", "UnitPrice": "Price"})
df = df.dropna(subset=["CustomerID"])                         # guest checkouts can't be targeted
df = df[~df["Invoice"].astype(str).str.startswith("C")]       # drop cancellations
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["CustomerID"] = df["CustomerID"].astype(int)
df["Revenue"] = df["Quantity"] * df["Price"]

print(f"{df.CustomerID.nunique():,} customers | £{df.Revenue.sum():,.0f} revenue | "
      f"{df.InvoiceDate.min():%b %Y} to {df.InvoiceDate.max():%b %Y}")

In [ ]:
# 3. CUSTOMER VALUE + SEGMENTS
RISK_DAYS = 90      # no order in 90+ days = going quiet
LOST_DAYS = 365     # no order in a year = treat as lost
TOP_PCT   = 0.80    # top 20% of customers by lifetime spend = "high value"

snapshot = df.InvoiceDate.max() + pd.Timedelta(days=1)
c = df.groupby("CustomerID").agg(first=("InvoiceDate", "min"), last=("InvoiceDate", "max"),
                                 orders=("Invoice", "nunique"), revenue=("Revenue", "sum"))
c["recency_days"] = (snapshot - c["last"]).dt.days
c["tenure_days"]  = (c["last"] - c["first"]).dt.days
c["annual_value"] = c["revenue"] / np.maximum(c["tenure_days"], 365) * 365   # what they spend per year

high = c["revenue"] >= c["revenue"].quantile(TOP_PCT)
c["segment"] = np.select(
    [high & (c.recency_days <= RISK_DAYS),
     high & (c.recency_days > RISK_DAYS) & (c.recency_days <= LOST_DAYS),
     c.recency_days <= RISK_DAYS],
    ["Champions", "At-risk champions", "Other active"], default="Lapsed / lost")

seg = c.groupby("segment").agg(customers=("revenue", "size"), revenue=("revenue", "sum"),
                               annual_value=("annual_value", "sum"))
seg["% customers"] = (seg.customers / seg.customers.sum() * 100).round(1)
seg["% revenue"]   = (seg.revenue / seg.revenue.sum() * 100).round(1)
order = ["Champions", "At-risk champions", "Other active", "Lapsed / lost"]
seg = seg.reindex(order)
seg.style.format({"revenue": "£{:,.0f}", "annual_value": "£{:,.0f}"})

In [ ]:
# 4. THE HEADLINE NUMBER
risk = c[c.segment == "At-risk champions"]
at_risk_value = risk.annual_value.sum()
top_share = c[high].revenue.sum() / c.revenue.sum() * 100

print(f"Top 20% of customers drive {top_share:.0f}% of all revenue.")
print(f"{len(risk):,} of them ({len(risk)/len(c)*100:.1f}% of customers) haven't ordered in {RISK_DAYS}–{LOST_DAYS} days.")
print(f"They are worth £{at_risk_value:,.0f} a year. That is the revenue at risk.")

In [ ]:
# 5. CHARTS (saved as PNGs for your slides)
BLUE, ORANGE, GREY, INK, MUTED = "#2a78d6", "#eb6834", "#c9c8c2", "#0b0b0b", "#52514e"
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": GREY, "axes.labelcolor": MUTED, "xtick.color": MUTED,
                     "ytick.color": MUTED, "text.color": INK})

# Chart 1: yearly value by segment, at-risk highlighted
fig, ax = plt.subplots(figsize=(9, 4.5))
vals = seg["annual_value"] / 1000
colors = [ORANGE if s == "At-risk champions" else BLUE for s in vals.index]
bars = ax.barh(vals.index[::-1], vals.values[::-1], color=colors[::-1], height=0.55)
for b, v in zip(bars, vals.values[::-1]):
    ax.text(b.get_width(), b.get_y() + b.get_height()/2, f"  £{v:,.0f}k", va="center", color=INK)
ax.set_xlabel("Yearly customer value (£ thousands)")
ax.xaxis.grid(True, color="#ecebe7"); ax.set_axisbelow(True)
ax.set_title(f"£{at_risk_value/1000:,.0f}k a year sits with high-value customers who have gone quiet",
             loc="left", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig("chart1_value_by_segment.png", dpi=200); plt.show()

# Chart 2: monthly cohort retention
df["cohort"] = df.groupby("CustomerID")["InvoiceDate"].transform("min").dt.to_period("M")
df["order_month"] = df["InvoiceDate"].dt.to_period("M")
df["months_since"] = (df.order_month - df.cohort).apply(lambda x: x.n)
coh = df.groupby(["cohort", "months_since"])["CustomerID"].nunique().unstack(fill_value=0)
ret = (coh.div(coh[0], axis=0) * 100).iloc[:-1, 1:13]   # skip month 0 (always 100%) and the last, partial cohort
ret = ret.mask(ret.eq(0))   # blank = month not reached yet

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(ret.values, cmap="Blues", aspect="auto", vmin=0, vmax=50)
ax.set_xticks(range(ret.shape[1])); ax.set_xticklabels(ret.columns)
ax.set_yticks(range(len(ret))); ax.set_yticklabels([str(p) for p in ret.index], fontsize=8)
ax.set_xlabel("Months since first order"); ax.set_ylabel("First-order month")
ax.spines[:].set_visible(False)
cb = fig.colorbar(im, ax=ax, shrink=0.8); cb.set_label("% of cohort still ordering"); cb.outline.set_visible(False)
m1 = ret[1].mean()
ax.set_title(f"Just {m1:.0f}% of new customers order again the following month",
             loc="left", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig("chart2_cohort_retention.png", dpi=200); plt.show()

In [ ]:
# 6. BUSINESS CASE (assumptions are labelled; change them and re-run)
WIN_BACK_RATES = [0.05, 0.10, 0.15]   # ASSUMPTION: share of at-risk value recovered
OFFER_COST     = 0.10                 # ASSUMPTION: 10% discount given on recovered revenue

case = pd.DataFrame({"Scenario": ["Conservative", "Base", "Optimistic"],
                     "Win-back rate": [f"{r:.0%}" for r in WIN_BACK_RATES],
                     "Recovered revenue": [at_risk_value * r for r in WIN_BACK_RATES]})
case["Discount cost"] = case["Recovered revenue"] * OFFER_COST
case["Net gain / year"] = case["Recovered revenue"] - case["Discount cost"]
case.style.format({k: "£{:,.0f}" for k in ["Recovered revenue", "Discount cost", "Net gain / year"]}).hide(axis="index")

In [ ]:
# 7. EXPORT THE TARGET LIST (what Audience Agent would build) + slide summary
fav = (df[df.CustomerID.isin(risk.index)].groupby(["CustomerID", "Description"])["Revenue"].sum()
       .reset_index().sort_values("Revenue").groupby("CustomerID").tail(1).set_index("CustomerID")["Description"])
target = risk.assign(favourite_product=fav)[["annual_value", "recency_days", "orders", "favourite_product"]]
target = target.sort_values("annual_value", ascending=False).round(0)
target.to_csv("at_risk_champions.csv")
display(target.head(10))

base = case.loc[1, "Net gain / year"]
print(f"""
SLIDE 1 (Problem): Top 20% of customers = {top_share:.0f}% of revenue. {len(risk):,} of them have gone quiet.
SLIDE 2 (Insight): £{at_risk_value:,.0f}/year at risk. Just {m1:.0f}% of new customers reorder the following month.
SLIDE 3 (Solution): Adobe win-back journey. Base case: £{base:,.0f} net gain per year (10% win-back, 10% discount).
""")

## How Adobe would solve it
| Step | Adobe product | AI agent | What it does here |
|---|---|---|---|
| 1. Unify data | Real-Time CDP (Experience Platform) | | One profile per customer from orders, web and email |
| 2. Find the segment | Real-Time CDP | **Audience Agent** | Builds the at-risk segment from a plain-English prompt |
| 3. Build the journey | Journey Optimizer | **Journey Agent** | Designs and tunes the win-back journey |
| 4. Personalise content | GenStudio / Firefly | | Creative variants featuring each customer's favourite product |
| 5. Test the offer | Journey Optimizer Experimentation Accelerator | **Experimentation Agent** | Discount vs no discount, with a holdout group |
| 6. Prove the value | Customer Journey Analytics | **Data Insights Agent** | Recovered revenue vs the holdout group |

**Example prompts I'd demo:**
- *Audience Agent:* "Create an audience of customers in the top 20% by lifetime spend who haven't ordered in 90 to 365 days."
- *Journey Agent:* "Build a win-back journey: personalised email on day 0, reminder on day 7, offer on day 14, exit when they purchase."

**Why a holdout group matters:** some customers would come back anyway. Comparing against a group that gets nothing proves the uplift is real. That is the number a CFO will ask for.

## Limits (say these out loud in the interview)
- Win-back rate and discount cost are assumptions, not results.
- Data runs to 2011 and covers one retailer, so treat it as a method demo.
- "Yearly value" averages past spend. It doesn't predict future spend.